# SLM Evaluation Analysis

Three open-weight SLMs evaluated on three production-relevant tasks:
- **Structured Extraction** — field-level F1 on synthetic job postings
- **RAG Q&A** — exact match + F1 on SQuAD (passage provided, not open-domain)
- **Intent Classification** — accuracy on Banking77 (77-class)


In [10]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

RESULTS_DIR = Path("../results")
MODELS = ["qwen", "phi", "llama"]
PARAM_COUNTS = {"qwen": 0.5, "phi": 3.8, "llama": 3.0}  # billions

df = pd.read_csv(RESULTS_DIR / "summary.csv")
df.head()

,model,model_load_time_s,model_disk_size_gb,task,n_samples,f1,precision,recall,tokens_per_sec,mean_ttft_ms,mean_output_tokens,peak_memory_gb,exact_match,accuracy,json_parse_rate,faithfulness_rate,macro_f1
0,qwen,1.13,1.999,extraction,150,0.5637,0.6144,0.5281,27.12,40.9,58.8,1.001,NaN,NaN,0.1267,NaN,NaN
1,qwen,1.13,1.999,rag_qa,200,30.4214,NaN,NaN,35.34,30.3,17.6,1.942,4.5,NaN,NaN,0.055,NaN
2,qwen,1.13,1.999,classification,200,NaN,NaN,NaN,13.42,22.2,3.8,2.280,NaN,0.155,NaN,NaN,0.0309
3,phi,33.67,15.289,extraction,150,0.5614,0.5556,0.5719,7.66,402.2,84.2,0.423,NaN,NaN,0.0000,NaN,NaN
4,phi,33.67,15.289,rag_qa,200,69.1078,NaN,NaN,6.45,419.5,9.6,0.741,47.0,NaN,NaN,0.565,NaN


## 1. Accuracy by Model and Task

Primary metric per task: F1 for extraction and RAG Q&A, accuracy for classification.

In [ ]:
metric_map = {
    "extraction": "f1",
    "rag_qa": "f1",
    "classification": "accuracy",
}

rows = []
for _, row in df.iterrows():
    metric = metric_map[row["task"]]
    score = row[metric]
    if row["task"] == "rag_qa":
        score = score / 100  # rag_qa F1 is reported on 0-100 scale; normalize to 0-1
    rows.append({"model": row["model"], "task": row["task"], "score": score, "metric": metric})
plot_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_df, x="task", y="score", hue="model", ax=ax)
ax.set_title("Primary accuracy metric by model and task")
ax.set_ylabel("Score (F1 or Accuracy)")
ax.set_ylim(0, 1)

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f"{height:.2f}",
            ha="center", va="bottom", fontsize=8,
        )

plt.tight_layout()
plt.show()

## 2. Throughput vs. Accuracy

Each point is one model × task combination. Bubble size reflects peak memory usage (GB). Models in the upper-right with small bubbles offer the best quality-per-resource tradeoff.

In [ ]:
import matplotlib.lines as mlines

scatter_df = plot_df.merge(df[["model", "task", "tokens_per_sec", "peak_memory_gb"]], on=["model", "task"])

fig, ax = plt.subplots(figsize=(8, 6))
markers = ["o", "s", "^"]
legend_handles = []

for model, marker in zip(scatter_df["model"].unique(), markers):
    sub = scatter_df[scatter_df["model"] == model]
    sc = ax.scatter(
        sub["tokens_per_sec"],
        sub["score"],
        s=sub["peak_memory_gb"] * 400,
        label=model,
        marker=marker,
        alpha=0.8,
    )
    color = sc.get_facecolor()[0]
    legend_handles.append(
        mlines.Line2D([], [], color=color, marker=marker, linestyle="None", markersize=8, label=model)
    )

ax.set_xlabel("Tokens / second")
ax.set_ylabel("Score")
ax.set_title("Throughput vs. accuracy (bubble size = peak memory GB)")
ax.legend(handles=legend_handles)
plt.tight_layout()
plt.show()

## 3. Accuracy per Billion Parameters

Does bigger always mean better? Normalizing by parameter count, memory footprint, and throughput reveals which models deliver the most quality per unit of compute — the central question for edge deployment decisions.

In [ ]:
param_df = plot_df.copy()
param_df["params_b"] = param_df["model"].map(PARAM_COUNTS)
param_df = param_df.merge(df[["model", "task", "tokens_per_sec", "peak_memory_gb"]], on=["model", "task"])

param_df["score_per_param"] = param_df["score"] / param_df["params_b"]
param_df["score_per_gb"] = param_df["score"] / param_df["peak_memory_gb"]
param_df["score_per_tok_sec"] = param_df["score"] / param_df["tokens_per_sec"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.barplot(data=param_df, x="task", y="score_per_param", hue="model", ax=axes[0])
axes[0].set_title("Score / B params")
axes[0].set_ylabel("Score / B params")
axes[0].set_xlabel("")

sns.barplot(data=param_df, x="task", y="score_per_gb", hue="model", ax=axes[1])
axes[1].set_title("Score / GB memory")
axes[1].set_ylabel("Score / GB")
axes[1].set_xlabel("")
axes[1].get_legend().remove()

sns.barplot(data=param_df, x="task", y="score_per_tok_sec", hue="model", ax=axes[2])
axes[2].set_title("Score / (token/sec)")
axes[2].set_ylabel("Score / (tok/s)")
axes[2].set_xlabel("")
axes[2].get_legend().remove()

plt.suptitle("Accuracy per resource unit", y=1.02)
plt.tight_layout()
plt.show()

## 4. Time to First Token by Task

TTFT is the prefill cost — how long the model takes before it emits its first output token. It scales with both model size and prompt length. Classification has the longest prompts (77 intent labels + few-shot examples), making it the stress test for prefill latency. A high TTFT on classification is a real deployment concern for real-time intent routing.

In [ ]:
ttft_df = df[["model", "task", "mean_ttft_ms"]].copy()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=ttft_df, x="task", y="mean_ttft_ms", hue="model", ax=ax)
ax.set_title("Mean time to first token by task (ms)")
ax.set_ylabel("TTFT (ms)")

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + ax.get_ylim()[1] * 0.01,
            f"{height:.0f}",
            ha="center", va="bottom", fontsize=8,
        )

plt.tight_layout()
plt.show()

## 5. Extraction Deep-Dive

Per-field accuracy shows which extraction targets are consistently hard across models. JSON parse rate measures output reliability: a model that frequently falls back to regex extraction is less production-safe even if its F1 is comparable.

In [ ]:
FIELDS = ["job_title", "company", "location", "salary_range", "years_experience_required"]

per_field_rows = []
for model in MODELS:
    raw_path = RESULTS_DIR / "raw" / f"{model}_extraction.json"
    if not raw_path.exists():
        continue
    with open(raw_path) as f:
        data = json.load(f)
    for field in FIELDS:
        tp = sum(1 for r in data["raw"] if r["per_field"].get(field) == "tp")
        total = sum(1 for r in data["raw"] if r["per_field"].get(field) is not None)
        per_field_rows.append({"model": model, "field": field, "accuracy": tp / total if total > 0 else 0})

pf_df = pd.DataFrame(per_field_rows)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pf_df, x="field", y="accuracy", hue="model", ax=ax)
ax.set_title("Extraction: per-field accuracy by model")
ax.set_ylim(0, 1)
ax.set_xlabel("")

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f"{height:.2f}",
            ha="center", va="bottom", fontsize=8,
        )

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
if "json_parse_rate" in df.columns:
    parse_df = df[df["task"] == "extraction"][["model", "json_parse_rate"]].copy()

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=parse_df, x="model", y="json_parse_rate", ax=ax)
    ax.set_title("Extraction: JSON parse rate (no regex fallback needed)")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Parse rate")

    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.2f}",
                ha="center", va="bottom", fontsize=8,
            )

    plt.tight_layout()
    plt.show()
else:
    print("json_parse_rate not in summary.csv — re-run extraction evals to populate.")

## 6. RAG Q&A: Faithfulness

Exact match penalizes valid paraphrases. Faithfulness rate — the share of predictions that appear verbatim in the source passage — separates two failure modes: wrong-but-grounded (the model extracted something real from the context) vs. hallucinated (the answer isn't in the passage at all).

In [ ]:
if "faithfulness_rate" in df.columns:
    rag_df = df[df["task"] == "rag_qa"][["model", "exact_match", "faithfulness_rate"]].copy()
    rag_df["exact_match"] = rag_df["exact_match"] / 100  # normalize to 0-1
    rag_melted = rag_df.melt(
        id_vars="model",
        value_vars=["exact_match", "faithfulness_rate"],
        var_name="metric",
        value_name="score",
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=rag_melted, x="model", y="score", hue="metric", ax=ax)
    ax.set_title("RAG Q&A: exact match vs. faithfulness rate")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Rate (0–1)")

    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.2f}",
                ha="center", va="bottom", fontsize=8,
            )

    plt.tight_layout()
    plt.show()
else:
    print("faithfulness_rate not in summary.csv — re-run rag_qa evals to populate.")

## 7. Classification: Hardest Intents & Macro F1

Accuracy treats all 77 intents equally by sample count. Macro F1 weights each class equally regardless of frequency, penalizing models that are strong on common intents but fail on rare ones. A large accuracy–macro F1 gap indicates uneven coverage across the intent space.

In [ ]:
for model in MODELS:
    raw_path = RESULTS_DIR / "raw" / f"{model}_classification.json"
    if not raw_path.exists():
        continue
    with open(raw_path) as f:
        data = json.load(f)

    by_intent = {}
    for r in data["raw"]:
        intent = r["gold"]
        by_intent.setdefault(intent, {"correct": 0, "total": 0})
        by_intent[intent]["total"] += 1
        by_intent[intent]["correct"] += int(r["correct"])

    intent_acc = {k: v["correct"] / v["total"] for k, v in by_intent.items()}
    worst = sorted(intent_acc.items(), key=lambda x: x[1])[:10]

    print(f"\n{model} \u2014 10 hardest intents:")
    for intent, acc in worst:
        print(f"  {intent:<45} {acc:.0%}")

In [ ]:
if "macro_f1" in df.columns:
    clf_df = df[df["task"] == "classification"][["model", "accuracy", "macro_f1"]].copy()
    clf_melted = clf_df.melt(
        id_vars="model",
        value_vars=["accuracy", "macro_f1"],
        var_name="metric",
        value_name="score",
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=clf_melted, x="model", y="score", hue="metric", ax=ax)
    ax.set_title("Classification: accuracy vs. macro F1")
    ax.set_ylim(0, 1)

    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.2f}",
                ha="center", va="bottom", fontsize=8,
            )

    plt.tight_layout()
    plt.show()
else:
    print("macro_f1 not in summary.csv — re-run classification evals to populate.")

## 8. Key Findings

**Phi-3.5-mini is the strongest model overall, but carries a hidden prefill cost.** It nearly doubles Llama-3.2 on RAG Q&A (F1 69.1 vs 38.6) and leads on classification accuracy (74% vs 67%), but its mean time-to-first-token on classification is 2,210 ms — roughly 100× longer than Llama’s 22 ms — because the large model hits a long prompt (77 labels + few-shot examples) hard during prefill. For real-time intent routing, that cost is prohibitive.

**Qwen-0.5B punches above its weight on extraction, then hits a capability floor.** It delivers extraction F1 within 11 points of Llama at 1/6th the parameters, loads in under 1 second, and uses far less disk space — a compelling edge-deployment profile. But classification accuracy drops to 15.5%, indicating that very small models struggle with dense multi-class tasks that require reasoning over a long label list in context.

**Salary range is the hardest extraction field across all three models**, suggesting numeric ranges with irregular formatting benefit from post-processing heuristics rather than pure LM extraction.

**Llama-3.2-3B offers the best balance for latency-sensitive deployments**: competitive accuracy across all three tasks, 22 ms TTFT on classification, and a moderate disk footprint relative to phi.


## 9. Limitations & Next Steps

- **Prompt sensitivity:** Results are tied to these specific prompt templates; no prompt tuning was done. Small wording changes can shift accuracy meaningfully, especially for classification.
- **Few-shot count:** Banking77 used 5 fixed examples. Sweeping N (0, 5, 10, 20) would quantify how much few-shot helps each model and whether the TTFT cost scales linearly with example count.
- **Classification sample coverage:** 200 samples across 77 intents means many intents appear only 2–3 times. Per-intent accuracy estimates are noisy; the full 3,080-sample test set would give more reliable macro F1.
- **Quantization:** Running 4-bit quantized versions of the larger models would sharpen the accuracy-vs-memory tradeoff and make TTFT comparisons more practically relevant for CPU/edge deployments.
- **Faithfulness vs. exact match:** For RAG Q&A, faithfulness rate is a better proxy for hallucination risk in production pipelines than exact match, which penalizes valid paraphrases.
